In [1]:
import pandas as pd
import duckdb

raw = pd.read_parquet("../data/raw/steam_games.parquet",
                      columns=["appID", "name", "release_date", "genres",
                               "developers", "positive", "negative"])

# same filters as the cleaning notebook, minus the 10-review minimum
raw = raw[~raw["name"].str.contains("playtest", case=False, na=False)]

def is_empty(x):
    return x is None or len(x) == 0
raw = raw[~raw["genres"].apply(is_empty) & ~raw["developers"].apply(is_empty)]

software_genres = {"Game Development", "Education", "Design & Illustration",
                   "Animation & Modeling", "Utilities", "Software Training",
                   "Video Production", "Audio Production", "Photo Editing",
                   "Web Publishing", "Accounting"}
raw = raw[~raw["genres"].apply(lambda g: all(x in software_genres for x in g))]

raw["release_date"] = pd.to_datetime(raw["release_date"], format="%b %d, %Y", errors="coerce")
raw["year"] = raw["release_date"].dt.year
raw["total_reviews"] = raw["positive"] + raw["negative"]

print(raw.shape)
print("Latest release date:", raw["release_date"].max())

(114360, 9)
Latest release date: 2026-09-08 00:00:00


In [2]:
query = """
SELECT
    year,
    COUNT(*)                                                         AS releases,
    SUM(CASE WHEN total_reviews >= 10 THEN 1 ELSE 0 END)             AS reached_10_reviews,
    ROUND(AVG(CASE WHEN total_reviews >= 10 THEN 1 ELSE 0 END) * 100, 1) AS pct_reached_10
FROM raw
WHERE year >= 2006
GROUP BY year
ORDER BY year
"""
duckdb.sql(query).df()

,year,releases,reached_10_reviews,pct_reached_10
0,2006,69,66.0,95.7
1,2007,90,90.0,100.0
2,2008,155,148.0,95.5
3,2009,305,288.0,94.4
4,2010,253,240.0,94.9
5,2011,258,256.0,99.2
6,2012,317,316.0,99.7
7,2013,435,433.0,99.5
8,2014,1485,1470.0,99.0
9,2015,2455,2377.0,96.8


In [3]:
raw[raw["name"] == "Schedule I"][["name", "release_date", "positive", "negative", "total_reviews"]]

,name,release_date,positive,negative,total_reviews
99453,Schedule I,2025-03-24,200803,3238,204041


In [5]:
recent = raw[raw["release_date"] >= "2024-01-01"]
monthly = recent.groupby(recent["release_date"].dt.to_period("M")).agg(
    releases=("appID", "count"),
    pct_with_any_review=("total_reviews", lambda s: round((s > 0).mean() * 100, 1)),
)
monthly

,releases,pct_with_any_review
release_date,,
2024-01,1167,70.2
2024-02,1407,71.1
2024-03,1339,71.4
2024-04,1325,71.4
2024-05,1528,69.3
2024-06,1378,66.9
2024-07,1526,67.7
2024-08,1601,66.5
2024-09,1444,69.0


**Data issue found:** review counts are complete for games released through May 2025 (59–71% of each month's releases have reviews) but drop to ~3–6% from June 2025 onward, and to 0% for 2026. The dataset's review data appears to have been captured earlier than its game list (which runs to early Feb 2026). Decision: release-volume analysis uses full years 2006–2025; review-based analysis uses games released through 2024. The cleaned dataset excludes games released after May 31, 2025.